In [353]:
#overview


In [354]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [355]:
#load data
train_data=pd.read_csv("../data/raw/train.csv")
test_data=pd.read_csv("../data/raw/test.csv")

In [356]:
#exploratory data analysis

#for alley NA means no alley access,
#for BsmtQual NA means no basement,
#for poolqc NA means no pool, 
#for fence NA means no fence, 
#for miscfeature NA means no misc feature. 
#For the rest of the features, NA means missing value.

train_data['Alley']=train_data['Alley'].fillna('NoAlley')
train_data['BsmtQual']=train_data['BsmtQual'].fillna('NoBasement')
train_data['BsmtCond']=train_data['BsmtCond'].fillna('NoBasement')
train_data['BsmtExposure']=train_data['BsmtExposure'].fillna('NoBasement')
train_data['BsmtFinType1']=train_data['BsmtFinType1'].fillna('NoBasement')
train_data['BsmtFinType2']=train_data['BsmtFinType2'].fillna('NoBasement')
train_data['FireplaceQu']=train_data['FireplaceQu'].fillna('NoFireplace')
train_data['GarageType']=train_data['GarageType'].fillna('NoGarage')
train_data['GarageFinish']=train_data['GarageFinish'].fillna('NoGarage')
train_data['GarageQual']=train_data['GarageQual'].fillna('NoGarage')
train_data['GarageCond']=train_data['GarageCond'].fillna('NoGarage')
train_data['PoolQC']=train_data['PoolQC'].fillna('NoPool')
train_data['Fence']=train_data['Fence'].fillna('NoFence')
train_data['MiscFeature']=train_data['MiscFeature'].fillna('None')

train_data=train_data.dropna(subset=['Electrical'])

train_data[train_data["MasVnrType"].isna()][["MasVnrType","MasVnrArea"]]

train_data.loc[
    train_data["MasVnrType"].isna() & (train_data["MasVnrArea"] == 0),
    "MasVnrType"
] = "None"

train_data.loc[
    train_data["MasVnrType"].isna() & (train_data["MasVnrArea"].isna()),
    "MasVnrType"
] = "None"

train_data.loc[
    (train_data["MasVnrType"]=="None") & (train_data["MasVnrArea"].isna()),
    "MasVnrArea"
] = 0

train_data=train_data.dropna(subset=["MasVnrType"])

train_data["LotFrontage"] = train_data["LotFrontage"].fillna(
    train_data.groupby("Neighborhood")["LotFrontage"].transform("median")
)

train_data["HasGarage"] = (train_data["GarageType"] != "NoGarage").astype(int)

nn=train_data.isnull().sum()
print(nn[nn>0])
#no more missing values in the training data


GarageYrBlt    81
dtype: int64


In [357]:
#feature engineering
#after creating new features the results didnt change because the model already had the information
#in the original features

"""train_data["TotalSF"]=(
    train_data["TotalBsmtSF"]
    + train_data["1stFlrSF"]
    + train_data["2ndFlrSF"]
)


train_data["HouseAge"]=(
    train_data["YrSold"]-train_data["YearBuilt"]
)

train_data["RemodAge"]=(
    train_data["YrSold"]-train_data["YearRemodAdd"]
)

train_data["TotalOutdoorSF"]=(
    train_data["WoodDeckSF"]
    + train_data["OpenPorchSF"]
    + train_data["EnclosedPorch"]
    + train_data["3SsnPorch"]
    + train_data["ScreenPorch"]
)

train_data["TotalBath"]=(
    train_data["FullBath"]
    + 0.5 * train_data["HalfBath"]
    + train_data["BsmtFullBath"]
    + 0.5 * train_data["BsmtHalfBath"]
)
#for test
test_data["TotalSF"]=(
    test_data["TotalBsmtSF"]
    + test_data["1stFlrSF"]
    + test_data["2ndFlrSF"]
)


test_data["HouseAge"]=(
    test_data["YrSold"]-test_data["YearBuilt"]
)

test_data["RemodAge"]=(
    test_data["YrSold"]-test_data["YearRemodAdd"]
)

test_data["TotalOutdoorSF"]=(
    test_data["WoodDeckSF"]
    + test_data["OpenPorchSF"]
    + test_data["EnclosedPorch"]
    + test_data["3SsnPorch"]
    + test_data["ScreenPorch"]
)


test_data["TotalBath"]=(
    test_data["FullBath"]
    + 0.5 * test_data["HalfBath"]
    + test_data["BsmtFullBath"]
    + 0.5 * test_data["BsmtHalfBath"]
)
"""

'train_data["TotalSF"]=(\n    train_data["TotalBsmtSF"]\n    + train_data["1stFlrSF"]\n    + train_data["2ndFlrSF"]\n)\n\n\ntrain_data["HouseAge"]=(\n    train_data["YrSold"]-train_data["YearBuilt"]\n)\n\ntrain_data["RemodAge"]=(\n    train_data["YrSold"]-train_data["YearRemodAdd"]\n)\n\ntrain_data["TotalOutdoorSF"]=(\n    train_data["WoodDeckSF"]\n    + train_data["OpenPorchSF"]\n    + train_data["EnclosedPorch"]\n    + train_data["3SsnPorch"]\n    + train_data["ScreenPorch"]\n)\n\ntrain_data["TotalBath"]=(\n    train_data["FullBath"]\n    + 0.5 * train_data["HalfBath"]\n    + train_data["BsmtFullBath"]\n    + 0.5 * train_data["BsmtHalfBath"]\n)\n#for test\ntest_data["TotalSF"]=(\n    test_data["TotalBsmtSF"]\n    + test_data["1stFlrSF"]\n    + test_data["2ndFlrSF"]\n)\n\n\ntest_data["HouseAge"]=(\n    test_data["YrSold"]-test_data["YearBuilt"]\n)\n\ntest_data["RemodAge"]=(\n    test_data["YrSold"]-test_data["YearRemodAdd"]\n)\n\ntest_data["TotalOutdoorSF"]=(\n    test_data["WoodDeckS

In [358]:
#data preprocessing
x = train_data.drop(columns=["SalePrice", "Id"])
y = train_data["SalePrice"]
print(x.columns)

x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)


numerical_features = x.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x.select_dtypes(include=['object']).columns

#this pipeline is used to handle NaN values for garageyrblt
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

x_train_processed = preprocessor.fit_transform(x_train)
x_val_processed = preprocessor.transform(x_val)


Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

In [359]:
#linear regression model training and evaluation
model = LinearRegression()

model.fit(x_train_processed, y_train)
y_pred = model.predict(x_val_processed)
print("MAE:", mean_absolute_error(y_val, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("R²:", r2_score(y_val, y_pred))

MAE: 18986.059042732657
RMSE: 26633.530160125858
R²: 0.8788790773656282


In [360]:
reg_model=Ridge(alpha=10)
reg_model.fit(x_train_processed, y_train)
y_pred_reg = reg_model.predict(x_val_processed)
print("Ridge Regression MAE:", mean_absolute_error(y_val, y_pred_reg))
print("Ridge Regression RMSE:", np.sqrt(mean_squared_error(y_val, y_pred_reg)))
print("Ridge Regression R²:", r2_score(y_val, y_pred_reg))

#ridge is slightly better than linear

Ridge Regression MAE: 18425.649468365187
Ridge Regression RMSE: 24892.06848293768
Ridge Regression R²: 0.8942004850990906


In [361]:
#cross validation to find the best alpha for ridge regression
alphas = [0.01, 0.1, 1, 10, 100, 1000]
ridge_cv=RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(x_train_processed, y_train)
print("Best alpha:", ridge_cv.alpha_)
#best alpha is 10 overall

Best alpha: 10.0


In [362]:
#random forest model training and evaluation
model_rf=RandomForestRegressor(n_estimators=200, random_state=42)
model_rf.fit(x_train_processed, y_train)
y_pred_rf = model_rf.predict(x_val_processed)
print("Random Forest MAE:", mean_absolute_error(y_val, y_pred_rf))
print("Random Forest RMSE:", np.sqrt(mean_squared_error(y_val, y_pred_rf)))
print("Random Forest R²:", r2_score(y_val, y_pred_rf))

Random Forest MAE: 18150.737182130586
Random Forest RMSE: 26582.172739056514
Random Forest R²: 0.8793457418095566


In [363]:
gbr_model=GradientBoostingRegressor(n_estimators=300, learning_rate=0.2,max_depth=3, random_state=42)
gbr_model.fit(x_train_processed, y_train)
y_pred_gbr = gbr_model.predict(x_val_processed)
print("Gradient Boosting MAE:", mean_absolute_error(y_val, y_pred_gbr))
print("Gradient Boosting RMSE:", np.sqrt(mean_squared_error(y_val, y_pred_gbr)))
print("Gradient Boosting R²:", r2_score(y_val, y_pred_gbr))

#best model is gradient boosting regressor with n_estimators=300, learning_rate=0.2,max_depth=3 i will
#use it to predict the test data

Gradient Boosting MAE: 16209.15486506967
Gradient Boosting RMSE: 23526.423456260833
Gradient Boosting R²: 0.9054909230470728


In [ ]:
#cleaning the test data
test_data['Alley']=test_data['Alley'].fillna('NoAlley')
test_data['BsmtQual']=test_data['BsmtQual'].fillna('NoBasement')
test_data['BsmtCond']=test_data['BsmtCond'].fillna('NoBasement')
test_data['BsmtExposure']=test_data['BsmtExposure'].fillna('NoBasement')
test_data['BsmtFinType1']=test_data['BsmtFinType1'].fillna('NoBasement')
test_data['BsmtFinType2']=test_data['BsmtFinType2'].fillna('NoBasement')
test_data['FireplaceQu']=test_data['FireplaceQu'].fillna('NoFireplace')
test_data['GarageType']=test_data['GarageType'].fillna('NoGarage')
test_data['GarageFinish']=test_data['GarageFinish'].fillna('NoGarage')
test_data['GarageQual']=test_data['GarageQual'].fillna('NoGarage')
test_data['GarageCond']=test_data['GarageCond'].fillna('NoGarage')
test_data['PoolQC']=test_data['PoolQC'].fillna('NoPool')
test_data['Fence']=test_data['Fence'].fillna('NoFence')
test_data['MiscFeature']=test_data['MiscFeature'].fillna('None')


test_data[test_data["MasVnrType"].isna()][["MasVnrType","MasVnrArea"]]

test_data.loc[
    test_data["MasVnrType"].isna() & (test_data["MasVnrArea"] == 0),
    "MasVnrType"
] = "None"

test_data.loc[
    test_data["MasVnrType"].isna() & (test_data["MasVnrArea"].isna()),
    "MasVnrType"
] = "None"

test_data.loc[
    (test_data["MasVnrType"]=="None") & (test_data["MasVnrArea"].isna()),
    "MasVnrArea"
] = 0


test_data["LotFrontage"] = test_data["LotFrontage"].fillna(
    test_data.groupby("Neighborhood")["LotFrontage"].transform("median")
)

test_data["HasGarage"] = (test_data["GarageType"] != "NoGarage").astype(int)


nn=test_data.isnull().sum()
print(nn[nn>0])

MSZoning         4
Utilities        2
Exterior1st      1
Exterior2nd      1
MasVnrType       3
BsmtFinSF1       1
BsmtFinSF2       1
BsmtUnfSF        1
TotalBsmtSF      1
BsmtFullBath     2
BsmtHalfBath     2
KitchenQual      1
Functional       2
GarageYrBlt     78
GarageCars       1
GarageArea       1
SaleType         1
dtype: int64


In [ ]:
#train the model on the entire training data and then predict the test data to sibmit to kaggle
xx=train_data.drop(columns=["SalePrice", "Id"])
yy=train_data["SalePrice"]

numerical_features = xx.select_dtypes(include=['int64', 'float64']).columns
categorical_features = xx.select_dtypes(include=['object']).columns

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

xx_train_processed = preprocessor.fit_transform(xx)
xx_test_processed = preprocessor.transform(test_data.drop(columns=["Id"]))



In [ ]:
#model training and prediction
gbr_model=GradientBoostingRegressor(n_estimators=300, learning_rate=0.2,max_depth=3, random_state=42)
gbr_model.fit(xx_train_processed, yy)
y_pred_gbr = gbr_model.predict(xx_test_processed)

submission = pd.DataFrame({
    "Id": test_data["Id"],
    "SalePrice": y_pred_gbr
})

submission.to_csv("../submission.csv", index=False)

print(submission.shape)
print(submission.head())

(1459, 2)
     Id      SalePrice
0  1461  126180.997854
1  1462  163457.956468
2  1463  177997.020894
3  1464  193542.129263
4  1465  177533.563506


In [ ]:
# Log-transform the target variable results in better score
y_log = np.log1p(yy)
gbr_log = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.2,
    max_depth=3,
    random_state=42
)

gbr_log.fit(xx_train_processed, y_log)
pred_log = gbr_log.predict(xx_test_processed)
predictions = np.expm1(pred_log)
submission = pd.DataFrame({
    "Id": test_data["Id"],
    "SalePrice": predictions
})
submission.to_csv("../submission_log.csv", index=False)